In [1]:
# Install required packages (if necessary)
!pip install torch torchvision scikit-learn pillow

In [2]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Dataset
from torchvision import models, transforms
from sklearn.metrics import classification_report
from PIL import Image
from google.colab import drive

In [3]:
drive.mount('/content/drive')  # Mount Google Drive

data_dir = "/content/drive/MyDrive/AI-Engineer/CV/face-recognition/faces_data"  # Update with your dataset path

Mounted at /content/drive


In [4]:
# Define custom dataset
class FaceDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []
        self.classes = []

        class_to_idx = {}

        for file_name in os.listdir(root_dir):
            if file_name.endswith(('.jpg', '.jpeg', '.png')):
                class_label = '_'.join(file_name.split('_')[:-1])
                if class_label not in class_to_idx:
                    class_to_idx[class_label] = len(class_to_idx)
                self.image_paths.append(os.path.join(root_dir, file_name))
                self.labels.append(class_to_idx[class_label])

        self.classes = list(class_to_idx.keys())
        self.class_to_idx = class_to_idx

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert("RGB")
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label

In [5]:
# Define data transforms
data_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Load dataset
dataset = FaceDataset(root_dir=data_dir, transform=data_transforms)

In [6]:
# Split dataset into training and test sets
train_size = int(0.9 * len(dataset))
test_size = len(dataset) - train_size
# Seed di-set (42) supaya split train/test konsisten tiap kali notebook di-run ulang --
# tanpa ini, komposisi test set berubah setiap run dan akurasi yang dilaporkan jadi
# tidak reproducible.
generator = torch.Generator().manual_seed(42)
train_dataset, test_dataset = random_split(dataset, [train_size, test_size], generator=generator)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [7]:
next(iter(train_dataset))

(tensor([[[-1.2617, -1.2788, -1.3130,  ..., -0.2513, -0.4054, -0.4568],
          [-1.2959, -1.3130, -1.3130,  ..., -0.0629, -0.3883, -0.5938],
          [-1.3302, -1.3302, -1.3130,  ...,  0.2453, -0.2684, -0.6794],
          ...,
          [ 1.8550,  1.8550,  1.8722,  ..., -1.1247, -1.1418, -1.1589],
          [ 1.8722,  1.8550,  1.8550,  ..., -1.1247, -1.1247, -1.1247],
          [ 1.9064,  1.8722,  1.8379,  ..., -1.1247, -1.1247, -1.1075]],
 
         [[-1.4755, -1.4930, -1.5105,  ..., -0.6527, -0.8102, -0.8627],
          [-1.5105, -1.5105, -1.4930,  ..., -0.4601, -0.7927, -1.0028],
          [-1.5280, -1.5105, -1.4755,  ..., -0.1450, -0.6702, -1.0903],
          ...,
          [ 1.9209,  1.9559,  1.9559,  ..., -1.0903, -1.1078, -1.1253],
          [ 1.9734,  1.9734,  1.9734,  ..., -1.0903, -1.0903, -1.0903],
          [ 2.0259,  1.9909,  1.9559,  ..., -1.0903, -1.0903, -1.0728]],
 
         [[-1.2816, -1.2990, -1.2990,  ..., -0.6018, -0.7587, -0.8110],
          [-1.3164, -1.3164,

In [8]:
dataset.class_to_idx

{'Andy Samberg': 0,
 'Robert Downey Jr': 1,
 'Hrithik Roshan': 2,
 'Vijay Deverakonda': 3,
 'Brad Pitt': 4,
 'Hugh Jackman': 5,
 'Marmik': 6,
 'Zac Efron': 7,
 'Roger Federer': 8,
 'Henry Cavill': 9,
 'Tom Cruise': 10,
 'Kashyap': 11,
 'Amitabh Bachchan': 12,
 'Dwayne Johnson': 13,
 'Virat Kohli': 14,
 'Akshay Kumar': 15}

In [9]:
# len(dataset.class_to_idx)

In [10]:
# Load EfficientNetB0 model
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
num_classes = len(dataset.classes)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 216MB/s]


In [11]:
def train_model(model, criterion, optimizer, num_epochs=50):
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct_preds = 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * inputs.size(0)
            correct_preds += torch.sum(preds == labels.data)

        epoch_loss = running_loss / len(train_dataset)
        epoch_acc = correct_preds.double() / len(train_dataset)
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.4f}")

    print('Training complete')
    return model

In [12]:
# Train the model
model = train_model(model, criterion, optimizer)

Epoch 1/50, Loss: 1.3949, Accuracy: 0.6060
Epoch 2/50, Loss: 0.2196, Accuracy: 0.9487
Epoch 3/50, Loss: 0.1340, Accuracy: 0.9598
Epoch 4/50, Loss: 0.1073, Accuracy: 0.9667
Epoch 5/50, Loss: 0.1034, Accuracy: 0.9667
Epoch 6/50, Loss: 0.0686, Accuracy: 0.9812
Epoch 7/50, Loss: 0.0768, Accuracy: 0.9778
Epoch 8/50, Loss: 0.0935, Accuracy: 0.9709
Epoch 9/50, Loss: 0.0402, Accuracy: 0.9889
Epoch 10/50, Loss: 0.0449, Accuracy: 0.9855
Epoch 11/50, Loss: 0.0238, Accuracy: 0.9923
Epoch 12/50, Loss: 0.0304, Accuracy: 0.9915
Epoch 13/50, Loss: 0.0201, Accuracy: 0.9940
Epoch 14/50, Loss: 0.0102, Accuracy: 0.9983
Epoch 15/50, Loss: 0.0255, Accuracy: 0.9915
Epoch 16/50, Loss: 0.0280, Accuracy: 0.9932
Epoch 17/50, Loss: 0.0346, Accuracy: 0.9906
Epoch 18/50, Loss: 0.0270, Accuracy: 0.9932
Epoch 19/50, Loss: 0.0374, Accuracy: 0.9889
Epoch 20/50, Loss: 0.0373, Accuracy: 0.9897
Epoch 21/50, Loss: 0.0468, Accuracy: 0.9872
Epoch 22/50, Loss: 0.0608, Accuracy: 0.9795
Epoch 23/50, Loss: 0.0372, Accuracy: 0.99

In [13]:
# Save the model to Google Drive
torch.save(model.state_dict(), '/content/drive/MyDrive/AI-Engineer/CV/face-recognition/full_classification_model.pth')

# Save the backbone model
backbone = nn.Sequential(*list(model.children())[:-1])
torch.save(backbone.state_dict(), '/content/drive/MyDrive/AI-Engineer/CV/face-recognition/backbone_model.pth')

# Save the backbone model murni
torch.save(model.classifier.state_dict(), '/content/drive/MyDrive/AI-Engineer/CV/face-recognition/classifier_backbone_model.pth')
torch.save(model.features.state_dict(), '/content/drive/MyDrive/AI-Engineer/CV/face-recognition/pure_backbone_model.pth')

# Evaluate model
model.eval()
all_preds = []
all_labels = []
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Print classification report
print("Classification Report:")
print(classification_report(all_labels, all_preds, target_names=[str(cls) for cls in dataset.classes]))

Classification Report:
                   precision    recall  f1-score   support

     Andy Samberg       1.00      0.86      0.92         7
 Robert Downey Jr       1.00      1.00      1.00         6
   Hrithik Roshan       0.92      0.92      0.92        13
Vijay Deverakonda       0.86      1.00      0.92        12
        Brad Pitt       1.00      1.00      1.00        17
     Hugh Jackman       0.89      1.00      0.94         8
           Marmik       1.00      1.00      1.00         5
        Zac Efron       0.88      1.00      0.93         7
    Roger Federer       1.00      1.00      1.00        10
     Henry Cavill       1.00      0.85      0.92        13
       Tom Cruise       1.00      0.78      0.88         9
          Kashyap       1.00      1.00      1.00         3
 Amitabh Bachchan       1.00      1.00      1.00         5
   Dwayne Johnson       1.00      1.00      1.00        11
      Virat Kohli       1.00      1.00      1.00         3
     Akshay Kumar       0.67    

---

**Face Recognition — EfficientNet-B0 Embeddings + pgvector**

Bagian dari eksplorasi belajar AI Engineering pribadi saya.

Blog: https://shaka-ai.hashnode.dev · GitHub: https://github.com/arielshakaramiro